# Oxford-IIIT Pet: a full-spectrum Computer Vision project (TensorFlow / Keras)

One dataset, **every major CV task**. The Oxford-IIIT Pet dataset (7,349 images, 37 cat/dog breeds) ships with breed labels *and* pixel-level trimaps, so we can derive bounding boxes from the masks and cover classification, localization and segmentation on the same images. This is the Keras twin of the PyTorch notebook: same project, same sections, so you can compare the two frameworks side by side.

| # | Section | CV concept |
|---|---------|-----------|
| 1 | Data & EDA | `tf.data`, TFDS, splits, class balance, image statistics |
| 2 | Joint augmentation | geometric aug applied consistently to image + mask + box |
| 3 | CNN from scratch | convolutions, BatchNorm, pooling, baseline |
| 4 | Transfer learning | ImageNet pretraining, freezing, cosine schedule, label smoothing, mixed precision |
| 5 | Evaluation | top-k, confusion matrix, error analysis, calibration (ECE) |
| 6 | Grad-CAM | model explainability with `GradientTape` |
| 7 | Semantic segmentation | U-Net with pretrained ResNet-50 encoder, Dice/CE loss, mIoU |
| 8 | Multi-task learning | one backbone -> class + box + mask, GIoU loss |
| 9 | Object detection | pretrained TF-Hub SSD vs. our box head |
| 10 | Modern architecture | fine-tune ConvNeXt-Tiny (transformer-inspired CNN) |
| 11 | Embeddings & retrieval | feature extraction, cosine search, t-SNE |
| 12 | Autoencoder | denoising, latent space interpolation |
| 13 | Robustness | accuracy under noise, blur, occlusion, low-res |
| 14 | Export & inference | `.keras`, SavedModel, TFLite, latency, predicting your own photo |

**Requirements:** TensorFlow >= 2.16 (ships Keras 3). **How to run:** Colab with a GPU runtime works well. Set `FAST_DEV = True` first to smoke-test the whole notebook on a tiny subset, then flip it to `False`. Expect roughly 40-70 minutes end-to-end on a T4 GPU (the first run also downloads ~800 MB of data).

In [ ]:
# Uncomment on Colab / a fresh environment
# !pip install -q "tensorflow>=2.16" tensorflow-datasets tensorflow-hub scikit-learn matplotlib pandas

import os, random, time, json, io, math, contextlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from keras import layers
import tensorflow_datasets as tfds
from sklearn.metrics import confusion_matrix, classification_report

print("tensorflow", tf.__version__, "| keras", keras.__version__)

In [ ]:
# ----------------------------- CONFIG -----------------------------
SEED = 42
FAST_DEV = False        # True = tiny subset + 1 epoch per stage (smoke test)
S = 224                 # image side length fed to every model
BATCH = 32              # lower to 16 if you hit a GPU out-of-memory error
AUTOTUNE = tf.data.AUTOTUNE
import warnings
warnings.filterwarnings("ignore", message=".*shuffle=True.*")   # tf.data pipelines shuffle themselves
tf.get_logger().setLevel("ERROR")                                   # silence AutoGraph "could not transform" chatter

keras.utils.set_random_seed(SEED)
gpus = tf.config.list_physical_devices("GPU")
for g_ in gpus:
    try: tf.config.experimental.set_memory_growth(g_, True)
    except Exception: pass
MIXED_PRECISION = len(gpus) > 0          # fp16 compute on GPU, fp32 outputs
if MIXED_PRECISION:
    keras.mixed_precision.set_global_policy("mixed_float16")
print("GPUs:", len(gpus), "| mixed precision:", MIXED_PRECISION)

def E(n):                                # epochs helper: 1 epoch when FAST_DEV
    return 1 if FAST_DEV else n

RESULTS = {}                             # every evaluation is logged here -> summary table at the end

## 1. Data & exploratory analysis

TFDS gives us, per image: the **breed** (37 classes), **species** (cat/dog) and a **trimap** mask (1 = pet, 2 = background, 3 = border). We remap the trimap to `0 = pet, 1 = background, 2 = border`.

The official **test** split is held out; a validation set is carved from `train` (85/15). *Never tune on the test set.*

In [ ]:
info = tfds.builder("oxford_iiit_pet").info
CLASSES = info.features["label"].names
N_CLASSES = len(CLASSES)
SPECIES_NAMES = info.features["species"].names
CAT_ID = [n.lower() for n in SPECIES_NAMES].index("cat")

# metadata-only pass (images are not decoded) -> label/species per example
meta = tfds.load("oxford_iiit_pet", split="train",
                 decoders={"image": tfds.decode.SkipDecoding(), "segmentation_mask": tfds.decode.SkipDecoding()})
lab_sp = np.array([(int(e["label"]), int(e["species"])) for e in tfds.as_numpy(meta)])
CLASS_IS_CAT = np.zeros(N_CLASSES, dtype=bool)
for l_, s_ in lab_sp:
    CLASS_IS_CAT[l_] = (s_ == CAT_ID)
print(f"classes={N_CLASSES} ({CLASS_IS_CAT.sum()} cat breeds / {(~CLASS_IS_CAT).sum()} dog breeds) | train examples={len(lab_sp)}")

### 2. `tf.data` pipeline with *joint* augmentation

For segmentation and localization the image, mask and box must be transformed **together**. We apply the same random flip and zoom-crop to image and mask (they are concatenated into one 4-channel tensor, cropped together, then split and resized: bilinear for the image, nearest for the mask). The bounding box is then **derived from the final mask**, so it stays correct under any geometric augmentation. Colour jitter touches only the image.

Stage 1 (decode + resize, **cached as uint8**) runs once; stage 2 (augmentation + box) re-runs every epoch.

In [ ]:
def stage1(e):
    img = tf.image.resize(e["image"], (S, S), method="bilinear")
    img = tf.cast(tf.round(tf.clip_by_value(img, 0.0, 255.0)), tf.uint8)
    m = tf.image.resize(tf.cast(e["segmentation_mask"], tf.float32), (S, S), method="nearest")
    m = tf.cast(tf.clip_by_value(m - 1.0, 0.0, 2.0), tf.uint8)       # 0 pet, 1 background, 2 border
    return {"image": img, "mask": m, "label": tf.cast(e["label"], tf.int32)}

def augment(img, mask):
    flip = tf.random.uniform(()) < 0.5
    img = tf.cond(flip, lambda: tf.image.flip_left_right(img), lambda: img)
    mask = tf.cond(flip, lambda: tf.image.flip_left_right(mask), lambda: mask)
    crop = tf.cast(tf.random.uniform((), 0.7, 1.0) * S, tf.int32)            # random zoom-in
    both = tf.concat([img, mask], axis=-1)                                    # (S,S,4): RGB + mask
    both = tf.image.random_crop(both, tf.stack([crop, crop, 4]))
    img = tf.image.resize(both[..., :3], (S, S), method="bilinear")
    mask = tf.image.resize(both[..., 3:], (S, S), method="nearest")
    x = img / 255.0                                                           # photometric jitter on [0,1]
    x = tf.image.random_brightness(x, 0.15)
    x = tf.image.random_contrast(x, 0.8, 1.2)
    x = tf.image.random_saturation(x, 0.8, 1.2)
    img = tf.clip_by_value(x, 0.0, 1.0) * 255.0
    return img, mask

def mask_to_bbox(mask):
    fg = tf.squeeze(mask, -1) != 1                                            # pet or border
    ys = tf.where(tf.reduce_any(fg, axis=1))[:, 0]
    xs = tf.where(tf.reduce_any(fg, axis=0))[:, 0]
    def box():
        return tf.stack([tf.reduce_min(xs), tf.reduce_min(ys), tf.reduce_max(xs) + 1, tf.reduce_max(ys) + 1])
    full = lambda: tf.constant([0, 0, S, S], dtype=tf.int64)
    b = tf.cond(tf.size(xs) > 0, box, full)
    return tf.cast(b, tf.float32) / S                                         # x1,y1,x2,y2 in [0,1]

def stage2(d, train):
    img = tf.cast(d["image"], tf.float32); mask = tf.cast(d["mask"], tf.float32)
    if train:
        img, mask = augment(img, mask)
    mask = tf.cast(tf.round(mask), tf.int32)                                  # (S,S,1)
    return {"image": img, "mask": mask, "label": d["label"], "bbox": mask_to_bbox(mask)}

def make_ds(split, train, take=None):
    ds = tfds.load("oxford_iiit_pet", split=split, shuffle_files=False)
    if take: ds = ds.take(take)
    ds = ds.map(stage1, num_parallel_calls=AUTOTUNE).cache()
    n = 0
    for _ in ds: n += 1                                                        # also fills the cache
    ds = ds.apply(tf.data.experimental.assert_cardinality(n))                  # known length -> proper progress bars / epoch ends
    if train: ds = ds.shuffle(1000, seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(lambda d: stage2(d, train), num_parallel_calls=AUTOTUNE)
    steps = n // BATCH if train else math.ceil(n / BATCH)
    return ds.batch(BATCH, drop_remainder=train).prefetch(AUTOTUNE), n, steps

t_ = (128, 64, 64) if FAST_DEV else (None, None, None)
train_ds, n_train, STEPS = make_ds("train[:85%]", True, t_[0])
val_ds, n_val, _ = make_ds("train[85%:]", False, t_[1])
test_ds, n_test, _ = make_ds("test", False, t_[2])
print("train/val/test sizes:", n_train, n_val, n_test, "| steps per epoch:", STEPS)
val_labels = np.concatenate([d["label"].numpy() for d in val_ds])
print(f"validation set covers {len(set(val_labels.tolist()))}/{N_CLASSES} breeds (should be ~37 unless FAST_DEV)")

def for_cls(ds): return ds.map(lambda d: (d["image"], tf.one_hot(d["label"], N_CLASSES)), AUTOTUNE)
def for_seg(ds): return ds.map(lambda d: (d["image"], d["mask"]), AUTOTUNE)
def for_mt(ds):
    return ds.map(lambda d: (d["image"], {"cls": tf.one_hot(d["label"], N_CLASSES), "box": d["bbox"], "seg": d["mask"]}), AUTOTUNE)

b = next(iter(train_ds))
print({k: (tuple(v.shape), v.dtype.name) for k, v in b.items()})

In [ ]:
def collect(ds):
    out = {"image": [], "mask": [], "bbox": [], "label": []}
    for d in ds:
        out["image"].append(d["image"].numpy().astype(np.uint8)); out["mask"].append(d["mask"].numpy()[..., 0].astype(np.uint8))
        out["bbox"].append(d["bbox"].numpy()); out["label"].append(d["label"].numpy())
    return {k: np.concatenate(v) for k, v in out.items()}

test_np = collect(test_ds)     # un-augmented test set as numpy: used for visualisation, robustness, retrieval ...
print({k: v.shape for k, v in test_np.items()})

# ---------------- EDA: class balance, image sizes, pixel balance ----------------
raw = tfds.load("oxford_iiit_pet", split="train", shuffle_files=False).take(300)
sizes, pix = [], np.zeros(3)
for e in tfds.as_numpy(raw):
    sizes.append(e["image"].shape[:2][::-1])
    pix += np.bincount(e["segmentation_mask"].ravel(), minlength=4)[1:4]
w, h = zip(*sizes)
counts = np.bincount(lab_sp[:, 0], minlength=N_CLASSES)

fig, ax = plt.subplots(1, 3, figsize=(18, 5))
order = np.argsort(counts)
ax[0].barh(np.array(CLASSES)[order], counts[order]); ax[0].set_title("Images per breed (train)"); ax[0].tick_params(labelsize=7)
ax[1].scatter(w, h, s=8, alpha=0.5); ax[1].set_title("Original image sizes (px)"); ax[1].set_xlabel("width"); ax[1].set_ylabel("height")
ax[2].bar(["pet", "background", "border"], pix / pix.sum()); ax[2].set_title("Pixel share per segmentation class")
plt.tight_layout(); plt.show()
print(f"breed counts: min={counts.min()} max={counts.max()}  -> nearly balanced")
print("pixel share pet/background/border:", np.round(pix / pix.sum(), 3), "-> segmentation is imbalanced (border is tiny)")

In [ ]:
SEG_COLORS = np.array([[230, 60, 60], [40, 40, 60], [250, 210, 60]], dtype=np.uint8)   # pet, background, border

def draw_box(ax, box, color, lw=2):
    x1, y1, x2, y2 = [float(v) * S for v in box]
    ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor=color, linewidth=lw))

def show_samples(imgs, masks, boxes, labels, title=""):
    n = len(imgs); fig, axes = plt.subplots(2, n, figsize=(2.6 * n, 5.4))
    for j in range(n):
        axes[0, j].imshow(imgs[j].astype(np.uint8)); draw_box(axes[0, j], boxes[j], "lime")
        axes[0, j].set_title(CLASSES[int(labels[j])], fontsize=8); axes[0, j].axis("off")
        axes[1, j].imshow(SEG_COLORS[np.squeeze(masks[j])]); axes[1, j].axis("off")
    fig.suptitle(title); plt.tight_layout(); plt.show()

n_ = min(6, len(test_np["image"]))
show_samples(test_np["image"][:n_], test_np["mask"][:n_], test_np["bbox"][:n_], test_np["label"][:n_],
             "Test samples: image + derived box (green), trimap below")

# the SAME image pushed through the augmentation 6 times
one = next(iter(tfds.load("oxford_iiit_pet", split="train", shuffle_files=False).take(1)))
d1 = stage1(one)
augs = [stage2(d1, True) for _ in range(6)]
show_samples([a["image"].numpy() for a in augs], [a["mask"].numpy() for a in augs], [a["bbox"].numpy() for a in augs],
             [a["label"].numpy() for a in augs], "SAME image, 6 random augmentations (box + mask follow the image)")

## 3. Helpers: schedules, history, common layers

Keras gives us the training engine (`model.fit`) for free; we only need a few helpers: a cosine LR schedule, a history merger/plotter, and a custom layer that bakes ResNet's ImageNet preprocessing **into the model** (so the model always takes raw 0-255 RGB, which makes export and inference foolproof).

**Concepts:** LR schedules (cosine decay), AdamW / weight decay, mixed precision, preprocessing parity between training and serving.

In [ ]:
@keras.saving.register_keras_serializable(package="cv")
class ResNetPreprocess(layers.Layer):
    # RGB 0-255 -> BGR, ImageNet-mean-subtracted ("caffe" mode used by keras.applications.ResNet50)
    def call(self, x):
        x = tf.reverse(x, axis=[-1])
        return x - tf.constant([103.939, 116.779, 123.68], dtype=x.dtype)

def cosine_opt(lr, epochs, wd=1e-4):
    sched = keras.optimizers.schedules.CosineDecay(lr, decay_steps=max(1, STEPS * epochs), alpha=0.01)
    return keras.optimizers.AdamW(learning_rate=sched, weight_decay=wd)

def merge_hist(*hs):
    out = {}
    for h in hs:
        for k, v in h.history.items(): out.setdefault(k, []).extend(v)
    return out

def plot_history(hist, metrics, title=""):
    metrics = [m for m in metrics if m in hist]                    # tolerate metric-name differences between Keras versions
    if not metrics: print("no matching metrics to plot:", list(hist)); return
    fig, axes = plt.subplots(1, len(metrics), figsize=(5 * len(metrics), 3.5)); axes = np.atleast_1d(axes)
    for ax, m in zip(axes, metrics):
        ax.plot(hist[m], label="train")
        if f"val_{m}" in hist: ax.plot(hist[f"val_{m}"], label="val")
        ax.set_title(f"{title} {m}"); ax.set_xlabel("epoch"); ax.legend()
    plt.tight_layout(); plt.show()

def cls_metrics():
    return [keras.metrics.CategoricalAccuracy(name="acc"), keras.metrics.TopKCategoricalAccuracy(k=5, name="top5")]

def count_params(m): return m.count_params() / 1e6

## 4. Baseline: a CNN trained from scratch

A VGG-style stack: `[Conv-BN-ReLU] x2 -> MaxPool` repeated, then global average pooling and a softmax classifier. With only ~3k training images and 37 classes it will **overfit and plateau low**, and that is the point: it shows what pretraining buys you in the next section.

**Concepts:** convolution, receptive field, BatchNorm, ReLU, pooling, global average pooling, overfitting.

In [ ]:
def build_small_cnn(widths=(32, 64, 128, 256)):
    inp = keras.Input((S, S, 3)); x = layers.Rescaling(1 / 255.0)(inp)
    for w_ in widths:
        for _ in range(2):
            x = layers.Conv2D(w_, 3, padding="same", use_bias=False)(x); x = layers.BatchNormalization()(x); x = layers.ReLU()(x)
        x = layers.MaxPooling2D()(x)
    x = layers.GlobalAveragePooling2D()(x); x = layers.Dropout(0.3)(x)
    return keras.Model(inp, layers.Dense(N_CLASSES, activation="softmax", dtype="float32")(x), name="small_cnn")

scratch = build_small_cnn()
print(f"SmallCNN params: {count_params(scratch):.2f}M")
EP = E(15)
scratch.compile(optimizer=cosine_opt(2e-3, EP), loss=keras.losses.CategoricalCrossentropy(), metrics=cls_metrics())
h_scratch = scratch.fit(for_cls(train_ds), validation_data=for_cls(val_ds), epochs=EP,
                        callbacks=[keras.callbacks.EarlyStopping(monitor="val_acc", mode="max", patience=5, restore_best_weights=True)])
plot_history(h_scratch.history, ["loss", "acc"], "scratch")

## 5. Transfer learning with ResNet-50

Two-phase recipe that is the industry default:

1. **Feature extraction** - freeze the ImageNet backbone, train only the new head (fast, stable).
2. **Fine-tuning** - unfreeze the backbone with a small learning rate, cosine schedule, **label smoothing**.

Keras detail: we always call the backbone with `training=False` and keep its **BatchNorm layers frozen**, the standard recipe for fine-tuning on small datasets (otherwise BN statistics get corrupted by tiny batches).

**Concepts:** pretrained representations, freezing, BN in fine-tuning, label smoothing, regularization (dropout, weight decay).

In [ ]:
def set_backbone_trainable(base, flag):
    base.trainable = flag
    if flag:
        for l_ in base.layers:
            if isinstance(l_, layers.BatchNormalization): l_.trainable = False      # keep BN in inference mode

def build_transfer_classifier(weights="imagenet"):
    base = keras.applications.ResNet50(include_top=False, weights=weights, input_shape=(S, S, 3))
    inp = keras.Input((S, S, 3))
    x = ResNetPreprocess()(inp)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(N_CLASSES, name="logits")(x)
    out = layers.Softmax(dtype="float32", name="probs")(x)
    return keras.Model(inp, out, name="resnet50_cls"), base

clf, clf_base = build_transfer_classifier()
print(f"ResNet-50 classifier params: {count_params(clf):.1f}M")

# ---- phase 1: head only
set_backbone_trainable(clf_base, False)
clf.compile(optimizer=keras.optimizers.Adam(1e-3), loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1), metrics=cls_metrics())
h1 = clf.fit(for_cls(train_ds), validation_data=for_cls(val_ds), epochs=E(3))

# ---- phase 2: full fine-tune (BN frozen), cosine schedule
set_backbone_trainable(clf_base, True)
EP2 = E(8)
clf.compile(optimizer=cosine_opt(1e-4, EP2), loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1), metrics=cls_metrics())
h2 = clf.fit(for_cls(train_ds), validation_data=for_cls(val_ds), epochs=EP2,
             callbacks=[keras.callbacks.EarlyStopping(monitor="val_acc", mode="max", patience=3, restore_best_weights=True)])
plot_history(merge_hist(h1, h2), ["loss", "acc"], "ResNet-50")
clf.save("pet_resnet50_cls.keras")

## 6. Evaluation & error analysis

Accuracy alone is not enough. We report **top-1 / top-5**, a **cat-vs-dog** accuracy (do errors at least land in the right species?), **calibration (ECE)** (are confidences trustworthy?), a **confusion matrix**, the **most-confused breed pairs**, and the **most confident mistakes**.

In [ ]:
y_test = test_np["label"]

def ece_score(probs, y, bins=15):
    conf = probs.max(1); pred = probs.argmax(1); acc = (pred == y).astype(float)
    edges = np.linspace(0, 1, bins + 1); e = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any(): e += m.mean() * abs(acc[m].mean() - conf[m].mean())
    return float(e)

def evaluate_classifier(model, name):
    probs = model.predict(for_cls(test_ds), verbose=0).astype("float32")
    pred = probs.argmax(1)
    top1 = float((pred == y_test).mean())
    top5 = float((np.argsort(-probs, 1)[:, :5] == y_test[:, None]).any(1).mean())
    species = float((CLASS_IS_CAT[pred] == CLASS_IS_CAT[y_test]).mean())
    ece = ece_score(probs, y_test)
    RESULTS[name] = {"top1": top1, "top5": top5, "cat_vs_dog": species, "ECE": ece}
    print(f"{name:<22} top1 {top1:.4f} | top5 {top5:.4f} | cat-vs-dog {species:.4f} | ECE {ece:.4f}")
    return probs

_ = evaluate_classifier(scratch, "Scratch CNN (test)")
probs_clf = evaluate_classifier(clf, "ResNet-50 FT (test)")

In [ ]:
pred_clf = probs_clf.argmax(1); yt = y_test
cm = confusion_matrix(yt, pred_clf, labels=range(N_CLASSES))
cmn = cm / cm.sum(1, keepdims=True).clip(min=1)

fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(cmn, cmap="viridis"); plt.colorbar(im, fraction=0.046)
ax.set_xticks(range(N_CLASSES)); ax.set_yticks(range(N_CLASSES))
ax.set_xticklabels(CLASSES, rotation=90, fontsize=6); ax.set_yticklabels(CLASSES, fontsize=6)
ax.set_title("Normalized confusion matrix (test)"); ax.set_xlabel("predicted"); ax.set_ylabel("true")
plt.tight_layout(); plt.show()

off = cm.copy(); np.fill_diagonal(off, 0)
pairs = np.dstack(np.unravel_index(np.argsort(-off.ravel())[:8], off.shape))[0]
print("Most confused (true -> predicted):")
for t, p in pairs:
    if off[t, p] > 0: print(f"  {CLASSES[t]:<28} -> {CLASSES[p]:<28} x{off[t, p]}")

rep = classification_report(yt, pred_clf, labels=range(N_CLASSES), target_names=CLASSES, output_dict=True, zero_division=0)
worst = sorted(CLASSES, key=lambda c: rep[c]["f1-score"])[:6]
print("\nHardest breeds by F1:", [(c, round(rep[c]["f1-score"], 2)) for c in worst])

wrong = np.where(pred_clf != yt)[0]
show = wrong[np.argsort(-probs_clf.max(1)[wrong])][:8]
if len(show):
    fig, axes = plt.subplots(1, len(show), figsize=(2.6 * len(show), 3.4)); axes = np.atleast_1d(axes)
    for ax, i in zip(axes, show):
        ax.imshow(test_np["image"][i]); ax.axis("off")
        ax.set_title(f"true: {CLASSES[yt[i]]}\npred: {CLASSES[pred_clf[i]]} ({probs_clf[i].max():.2f})", fontsize=7)
    plt.suptitle("Most confident mistakes"); plt.tight_layout(); plt.show()

## 7. Explainability with Grad-CAM

*Where is the network looking?* Grad-CAM weights the last conv feature maps by the gradient of the class score, giving a coarse heatmap. In Keras we do this with `tf.GradientTape`: run the backbone, **watch** its output feature maps, run the head, and differentiate the class logit with respect to those maps. We also compare the heatmap for the **predicted** (wrong) class with the **true** class on misclassified images.

**Concepts:** gradient-based attribution, `GradientTape`, feature maps, spurious cues.

In [ ]:
pre_layer = ResNetPreprocess()
gap_layer, logits_layer = clf.get_layer("gap"), clf.get_layer("logits")

def gradcam(imgs, class_idx=None):
    imgs = tf.convert_to_tensor(imgs, tf.float32)
    feats = tf.cast(clf_base(pre_layer(imgs), training=False), tf.float32)          # (B,7,7,2048)
    with tf.GradientTape() as tape:
        tape.watch(feats)
        logits = logits_layer(gap_layer(feats))
        if class_idx is None: class_idx = tf.argmax(logits, axis=1)
        score = tf.gather(logits, tf.cast(class_idx, tf.int32), batch_dims=1)
    grads = tape.gradient(score, feats)
    w_ = tf.reduce_mean(grads, axis=(1, 2), keepdims=True)
    cam = tf.nn.relu(tf.reduce_sum(w_ * feats, axis=-1, keepdims=True))
    cam = tf.image.resize(cam, (S, S), method="bilinear")[..., 0]
    cam = cam - tf.reduce_min(cam, axis=(1, 2), keepdims=True)
    cam = cam / (tf.reduce_max(cam, axis=(1, 2), keepdims=True) + 1e-8)
    return cam.numpy(), tf.argmax(logits, 1).numpy()

def overlay(img, cam, alpha=0.45):
    return (1 - alpha) * (img / 255.0) + alpha * plt.cm.jet(cam)[..., :3]

rng = np.random.default_rng(1)
idxs = rng.choice(len(y_test), size=min(6, len(y_test)), replace=False)
cams, preds_ = gradcam(test_np["image"][idxs].astype("float32"))
fig, axes = plt.subplots(2, len(idxs), figsize=(2.6 * len(idxs), 5.4))
for j, i in enumerate(idxs):
    axes[0, j].imshow(test_np["image"][i]); axes[0, j].axis("off"); axes[0, j].set_title(CLASSES[yt[i]], fontsize=8)
    axes[1, j].imshow(overlay(test_np["image"][i], cams[j])); axes[1, j].axis("off"); axes[1, j].set_title(f"pred: {CLASSES[preds_[j]]}", fontsize=8)
plt.suptitle("Grad-CAM on random test images"); plt.tight_layout(); plt.show()

if len(wrong):
    ws = wrong[:4]; imgs_w = test_np["image"][ws].astype("float32")
    cam_p, _ = gradcam(imgs_w, pred_clf[ws]); cam_t, _ = gradcam(imgs_w, yt[ws])
    fig, axes = plt.subplots(3, len(ws), figsize=(2.8 * len(ws), 8.2)); axes = axes.reshape(3, -1)
    for j, i in enumerate(ws):
        axes[0, j].imshow(test_np["image"][i]); axes[0, j].set_title("input", fontsize=8)
        axes[1, j].imshow(overlay(test_np["image"][i], cam_p[j])); axes[1, j].set_title(f"evidence for WRONG: {CLASSES[pred_clf[i]]}", fontsize=7)
        axes[2, j].imshow(overlay(test_np["image"][i], cam_t[j])); axes[2, j].set_title(f"evidence for TRUE: {CLASSES[yt[i]]}", fontsize=7)
        for r in range(3): axes[r, j].axis("off")
    plt.tight_layout(); plt.show()

## 8. Semantic segmentation: U-Net with a pretrained encoder

Classification says *what*; segmentation says *which pixels*. We build a **U-Net**: the ResNet-50 is the encoder, exposing feature maps at 1/2, 1/4, 1/8, 1/16 and 1/32 resolution (taken from named layers such as `conv1_relu`, `conv2_block3_out`, ...), and a decoder upsamples back to full resolution, concatenating **skip connections** from the matching encoder stage.

* Loss = **cross-entropy + Dice** (Dice counters the class imbalance seen in EDA).
* Metrics = **pixel accuracy, per-class IoU, mIoU, Dice**.

**Concepts:** encoder-decoder, skip connections, multi-output backbones via `keras.Model(inputs, [layer outputs])`, Dice loss, IoU / mIoU.

In [ ]:
def build_resnet_encoder():
    base = keras.applications.ResNet50(include_top=False, weights="imagenet", input_shape=(S, S, 3))
    names = ["conv1_relu", "conv2_block3_out", "conv3_block4_out", "conv4_block6_out", "conv5_block3_out"]
    enc = keras.Model(base.input, [base.get_layer(n).output for n in names], name="resnet50_encoder")   # 112,56,28,14,7
    set_backbone_trainable(enc, True)
    return enc

def conv_bn_relu(x, f):
    x = layers.Conv2D(f, 3, padding="same", use_bias=False)(x); x = layers.BatchNormalization()(x)
    return layers.ReLU()(x)

def up_block(x, skip, f):
    x = layers.UpSampling2D(2, interpolation="bilinear")(x)
    x = layers.Concatenate()([x, skip])
    return conv_bn_relu(conv_bn_relu(x, f), f)

def decode(feats):
    s0, s1, s2, s3, s4 = feats
    d = up_block(s4, s3, 512); d = up_block(d, s2, 256); d = up_block(d, s1, 128); d = up_block(d, s0, 64)
    d = layers.UpSampling2D(2, interpolation="bilinear")(d)
    d = conv_bn_relu(d, 32)
    return layers.Conv2D(3, 1, activation="softmax", dtype="float32", name="seg")(d)

def build_unet():
    inp = keras.Input((S, S, 3)); x = ResNetPreprocess()(inp)
    enc = build_resnet_encoder()
    return keras.Model(inp, decode(enc(x, training=False)), name="resnet_unet")

@keras.saving.register_keras_serializable(package="cv")
def dice_loss(y_true, y_pred, eps=1e-6):
    y_true = tf.one_hot(tf.squeeze(tf.cast(y_true, tf.int32), -1), 3); y_pred = tf.cast(y_pred, tf.float32)
    inter = tf.reduce_sum(y_true * y_pred, axis=(0, 1, 2)); den = tf.reduce_sum(y_true + y_pred, axis=(0, 1, 2))
    return 1.0 - tf.reduce_mean((2 * inter + eps) / (den + eps))

@keras.saving.register_keras_serializable(package="cv")
def seg_loss(y_true, y_pred):
    ce = keras.losses.sparse_categorical_crossentropy(tf.squeeze(tf.cast(y_true, tf.int32), -1), tf.cast(y_pred, tf.float32))
    return tf.reduce_mean(ce) + dice_loss(y_true, y_pred)

@keras.saving.register_keras_serializable(package="cv")
class SegMeanIoU(keras.metrics.MeanIoU):
    def __init__(self, num_classes=3, name="miou", **kw):
        super().__init__(num_classes=num_classes, name=name, **kw)
    def update_state(self, y_true, y_pred, sample_weight=None):
        y_true = tf.squeeze(tf.cast(y_true, tf.int32), -1); y_pred = tf.argmax(y_pred, axis=-1, output_type=tf.int32)
        return super().update_state(y_true, y_pred, sample_weight)

unet = build_unet()
print(f"U-Net params: {count_params(unet):.1f}M")
EP = E(10)
unet.compile(optimizer=cosine_opt(5e-4, EP), loss=seg_loss,
             metrics=[keras.metrics.SparseCategoricalAccuracy(name="pix_acc"), SegMeanIoU()])
h_seg = unet.fit(for_seg(train_ds), validation_data=for_seg(val_ds), epochs=EP,
                 callbacks=[keras.callbacks.EarlyStopping(monitor="val_miou", mode="max", patience=4, restore_best_weights=True)])
plot_history(h_seg.history, ["loss", "miou"], "U-Net")
unet.save("pet_unet.keras")

In [ ]:
SEG_NAMES = ["pet", "background", "border"]

def conf_to_metrics(conf):
    tp = np.diag(conf).astype(float); union = conf.sum(0) + conf.sum(1) - tp
    iou = tp / np.maximum(union, 1); dice = 2 * tp / np.maximum(conf.sum(0) + conf.sum(1), 1)
    return {"pixel_acc": tp.sum() / conf.sum(), "mIoU": iou.mean(), "mDice": dice.mean(),
            "per_class_iou": dict(zip(SEG_NAMES, np.round(iou, 4).tolist()))}

def eval_seg(predict_seg, ds):
    conf = np.zeros(9, dtype=np.int64)
    for d in ds:
        pred = predict_seg(d["image"].numpy()).argmax(-1)
        conf += np.bincount((d["mask"].numpy()[..., 0] * 3 + pred).ravel(), minlength=9)
    return conf_to_metrics(conf.reshape(3, 3))

seg_metrics = eval_seg(lambda x: unet.predict_on_batch(x), test_ds)
RESULTS["U-Net (test)"] = {k: v for k, v in seg_metrics.items() if k != "per_class_iou"}
print(seg_metrics)

def show_seg_predictions(predict_seg, idxs):
    x = test_np["image"][idxs].astype("float32"); pred = predict_seg(x).argmax(-1)
    fig, axes = plt.subplots(4, len(idxs), figsize=(2.6 * len(idxs), 10.4))
    for j, i in enumerate(idxs):
        img = test_np["image"][i]; cutout = np.where((pred[j] != 1)[..., None], img, 255).astype(np.uint8)   # background removal
        axes[0, j].imshow(img); axes[1, j].imshow(SEG_COLORS[test_np["mask"][i]])
        axes[2, j].imshow(SEG_COLORS[pred[j]]); axes[3, j].imshow(cutout)
        for r in range(4): axes[r, j].axis("off")
    for r, t in enumerate(["image", "ground truth", "prediction", "cut-out (bg removed)"]): axes[r, 0].set_title(t, fontsize=9, loc="left")
    plt.tight_layout(); plt.show()

show_seg_predictions(lambda x: unet.predict_on_batch(x), list(range(min(6, len(y_test)))))

## 9. Multi-task learning: classify + localize + segment with ONE backbone

A single shared ResNet-50 encoder feeds three heads:

* **classification head** (breed) - categorical cross-entropy with label smoothing
* **box head** (single-object localization) - predicts `(cx, cy, w, h)` through a sigmoid, converted to `x1,y1,x2,y2`; loss = **L1 + GIoU**
* **segmentation decoder** (U-Net) - cross-entropy + Dice

Keras makes multi-task training declarative: a model with a dict of outputs, plus dicts of losses, loss weights and metrics keyed by output name.

**Concepts:** multi-task learning, loss weighting, bounding-box regression, IoU / GIoU, parametrising boxes.

In [ ]:
def _box_stats(a, b):
    lt = tf.maximum(a[:, :2], b[:, :2]); rb = tf.minimum(a[:, 2:], b[:, 2:])
    wh = tf.maximum(rb - lt, 0.0); inter = wh[:, 0] * wh[:, 1]
    area = lambda t: tf.maximum(t[:, 2] - t[:, 0], 0.0) * tf.maximum(t[:, 3] - t[:, 1], 0.0)
    union = area(a) + area(b) - inter + 1e-7
    return inter / union, union

@keras.saving.register_keras_serializable(package="cv")
def iou(y_true, y_pred):
    return _box_stats(tf.cast(y_true, tf.float32), tf.cast(y_pred, tf.float32))[0]

@keras.saving.register_keras_serializable(package="cv")
def box_loss(y_true, y_pred):
    a = tf.cast(y_pred, tf.float32); b = tf.cast(y_true, tf.float32)
    iou, union = _box_stats(a, b)
    lt_c = tf.minimum(a[:, :2], b[:, :2]); rb_c = tf.maximum(a[:, 2:], b[:, 2:])
    wh_c = tf.maximum(rb_c - lt_c, 0.0); area_c = wh_c[:, 0] * wh_c[:, 1] + 1e-7
    giou = iou - (area_c - union) / area_c
    return 5.0 * tf.reduce_mean(tf.abs(a - b), axis=-1) + (1.0 - giou)

@keras.saving.register_keras_serializable(package="cv")
class CxCyWhToXyxy(layers.Layer):
    def call(self, t):
        t = tf.cast(t, tf.float32); cx, cy, bw, bh = tf.unstack(t, axis=-1)
        return tf.clip_by_value(tf.stack([cx - bw / 2, cy - bh / 2, cx + bw / 2, cy + bh / 2], -1), 0.0, 1.0)

def build_multitask():
    inp = keras.Input((S, S, 3)); x = ResNetPreprocess()(inp)
    enc = build_resnet_encoder()
    feats = enc(x, training=False)
    pooled = layers.GlobalAveragePooling2D()(feats[-1])
    cls = layers.Dense(N_CLASSES)(layers.Dropout(0.2)(pooled)); cls = layers.Softmax(dtype="float32", name="cls")(cls)
    b = layers.Dense(512, activation="relu")(pooled); b = layers.Dropout(0.1)(b)
    b = layers.Dense(4, activation="sigmoid", dtype="float32")(b); box = CxCyWhToXyxy(name="box", dtype="float32")(b)
    seg = decode(feats)                                              # named "seg"
    return keras.Model(inp, {"cls": cls, "box": box, "seg": seg}, name="pet_multitask")

multi = build_multitask()
print(f"multi-task params: {count_params(multi):.1f}M")
EP = E(12)
multi.compile(optimizer=cosine_opt(5e-4, EP),
              loss={"cls": keras.losses.CategoricalCrossentropy(label_smoothing=0.1), "box": box_loss, "seg": seg_loss},
              loss_weights={"cls": 1.0, "box": 1.0, "seg": 1.0},
              metrics={"cls": [keras.metrics.CategoricalAccuracy(name="acc")], "box": [iou], "seg": [SegMeanIoU()]})
h_mt = multi.fit(for_mt(train_ds), validation_data=for_mt(val_ds), epochs=EP,
                 callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True)])
print("logged keys:", list(h_mt.history.keys()))
plot_history(h_mt.history, ["loss", "cls_acc", "box_iou", "seg_miou"], "multi-task")
multi.save("pet_multitask.keras")

In [ ]:
def eval_multitask(model, ds):
    correct = n = 0; ious = []; conf = np.zeros(9, dtype=np.int64)
    for d in ds:
        out = model.predict_on_batch(d["image"].numpy())
        correct += int((out["cls"].argmax(1) == d["label"].numpy()).sum()); n += len(out["cls"])
        ious.append(_box_stats(tf.constant(out["box"], tf.float32), d["bbox"])[0].numpy())
        conf += np.bincount((d["mask"].numpy()[..., 0] * 3 + out["seg"].argmax(-1)).ravel(), minlength=9)
    ious = np.concatenate(ious); seg = conf_to_metrics(conf.reshape(3, 3))
    return {"top1": correct / n, "box_mean_IoU": float(ious.mean()), "box_IoU>0.5": float((ious > 0.5).mean()),
            "mIoU": seg["mIoU"], "pixel_acc": seg["pixel_acc"]}

mt_metrics = eval_multitask(multi, test_ds)
RESULTS["Multi-task (test)"] = mt_metrics
print(pd.Series(mt_metrics).round(4).to_string())
print("\nSingle-task vs multi-task")
print(f"  breed top-1 : ResNet-50 FT {RESULTS['ResNet-50 FT (test)']['top1']:.4f}  vs  multi-task {mt_metrics['top1']:.4f}")
print(f"  seg mIoU    : U-Net       {RESULTS['U-Net (test)']['mIoU']:.4f}  vs  multi-task {mt_metrics['mIoU']:.4f}")

def show_multitask(idxs):
    out = multi.predict_on_batch(test_np["image"][idxs].astype("float32"))
    fig, axes = plt.subplots(2, len(idxs), figsize=(2.8 * len(idxs), 5.8))
    for j, i in enumerate(idxs):
        p = int(out["cls"][j].argmax()); ok = p == int(test_np["label"][i])
        axes[0, j].imshow(test_np["image"][i]); draw_box(axes[0, j], test_np["bbox"][i], "lime"); draw_box(axes[0, j], out["box"][j], "red")
        axes[0, j].set_title(CLASSES[p], fontsize=8, color="green" if ok else "red")
        axes[1, j].imshow(SEG_COLORS[out["seg"][j].argmax(-1)])
        for r in range(2): axes[r, j].axis("off")
    plt.suptitle("Multi-task output: green box = GT, red box = prediction, title = breed"); plt.tight_layout(); plt.show()

show_multitask(list(range(min(6, len(y_test)))))

## 10. Object detection with a pretrained detector

Our box head predicts **one** box per image. Real detectors (SSD, Faster R-CNN, YOLO, DETR) predict **many** boxes + classes per image, trained on COCO with anchors / grid cells and NMS. COCO already knows `cat` (id 17) and `dog` (id 18), so we can run a pretrained **SSD MobileNet-V2 from TF Hub** zero-shot and compare its localization quality with our custom box head on the same test images.

(Wrapped in `try/except`: if TF Hub is unreachable the rest of the notebook still runs.)

**Concepts:** anchors, SSD, NMS, confidence thresholds, COCO, IoU@0.5.

In [ ]:
DET_OK = True
try:
    import tensorflow_hub as hub
    detector = hub.load("https://tfhub.dev/tensorflow/ssd_mobilenet_v2/2")
    _ = detector(tf.constant(test_np["image"][:1]))                # smoke test: this model takes ONE uint8 image [1,H,W,3] (no batching)
except Exception as e:
    DET_OK = False; print("Could not load/run the pretrained detector (offline / Hub change?) - skipping this section:", e)

if DET_OK:
    PET_IDS = {17, 18}                                               # COCO: 17 = cat, 18 = dog
    n_eval = min(120, len(y_test))
    gt_boxes = test_np["bbox"][:n_eval]
    mt_out = multi.predict(test_np["image"][:n_eval].astype("float32"), batch_size=16, verbose=0)
    mt_iou = _box_stats(tf.constant(mt_out["box"], tf.float32), tf.constant(gt_boxes, tf.float32))[0].numpy()
    det_iou, vis = [], []
    for i in range(n_eval):
        out = detector(tf.constant(test_np["image"][i][None]))       # one image at a time
        boxes = out["detection_boxes"][0].numpy(); scores = out["detection_scores"][0].numpy(); cls_ = out["detection_classes"][0].numpy().astype(int)
        keep = [j for j in range(len(scores)) if cls_[j] in PET_IDS and scores[j] > 0.3]
        if keep:
            j = max(keep, key=lambda q: scores[q]); ymin, xmin, ymax, xmax = boxes[j]       # Hub boxes are [ymin, xmin, ymax, xmax] in 0..1
            det_iou.append(float(_box_stats(tf.constant([[xmin, ymin, xmax, ymax]], tf.float32), tf.constant(gt_boxes[i:i + 1], tf.float32))[0][0]))
        else:
            det_iou.append(0.0)
        if len(vis) < 4: vis.append((i, boxes, scores, cls_))
    det_iou = np.array(det_iou)
    RESULTS["SSD zero-shot"] = {"box_mean_IoU": float(det_iou.mean()), "box_IoU>0.5": float((det_iou > 0.5).mean())}
    print(f"SSD MobileNet-V2 (COCO, zero-shot): mean IoU {det_iou.mean():.3f} | IoU>0.5 {(det_iou > 0.5).mean():.3f}")
    print(f"Our multi-task box head           : mean IoU {mt_iou.mean():.3f} | IoU>0.5 {(mt_iou > 0.5).mean():.3f}")
    fig, axes = plt.subplots(1, len(vis), figsize=(3 * len(vis), 3.4))
    for ax, (i, bx, sc, cl) in zip(np.atleast_1d(axes), vis):
        ax.imshow(test_np["image"][i]); ax.axis("off")
        for b_, s_, c_ in zip(bx, sc, cl):
            if s_ > 0.5:
                ymin, xmin, ymax, xmax = b_ * S
                ax.add_patch(plt.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin, fill=False, edgecolor="cyan", linewidth=2))
                ax.text(xmin, ymin, f"coco:{c_} {s_:.2f}", fontsize=7, backgroundcolor="cyan")
    plt.suptitle("Pretrained SSD detections (score > 0.5)"); plt.tight_layout(); plt.show()

## 11. A modern architecture: ConvNeXt-Tiny

`keras.applications` has no Vision Transformer, but it ships **ConvNeXt**: a pure CNN modernised with transformer-era design choices (large 7x7 depthwise kernels, LayerNorm instead of BatchNorm, GELU, inverted bottlenecks, AdamW + heavy regularisation). It makes a good "what changed since ResNet?" comparison. It has **no BatchNorm**, so we can fine-tune the whole network without the BN freezing trick, and it normalises its own inputs (it takes raw 0-255 RGB).

*Want the real ViT?* The `keras_hub` package provides ViT presets; the PyTorch notebook fine-tunes ViT-B/16 directly.

**Concepts:** architecture evolution (VGG -> ResNet -> ConvNeXt), LayerNorm vs BatchNorm, depthwise convolutions, large kernels.

In [ ]:
base_cnx = keras.applications.ConvNeXtTiny(include_top=False, weights="imagenet", input_shape=(S, S, 3), pooling="avg")
inp = keras.Input((S, S, 3)); x = base_cnx(inp); x = layers.Dropout(0.2)(x)
convnext = keras.Model(inp, layers.Dense(N_CLASSES, activation="softmax", dtype="float32")(x), name="convnext_cls")
print(f"ConvNeXt-Tiny classifier params: {count_params(convnext):.1f}M")
EP = E(5)
convnext.compile(optimizer=cosine_opt(1e-4, EP, wd=0.05), loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1), metrics=cls_metrics())
h_cnx = convnext.fit(for_cls(train_ds), validation_data=for_cls(val_ds), epochs=EP,
                     callbacks=[keras.callbacks.EarlyStopping(monitor="val_acc", mode="max", patience=3, restore_best_weights=True)])
plot_history(h_cnx.history, ["loss", "acc"], "ConvNeXt-T")
_ = evaluate_classifier(convnext, "ConvNeXt-T FT (test)")

## 12. Embeddings & image retrieval

Strip the classification layer and a CNN becomes a **feature extractor**. Cosine similarity between 2048-d embeddings gives *content-based image retrieval* ("find similar pets") with **no extra training**. We measure retrieval **precision@5** and visualize the embedding space with **t-SNE**.

**Concepts:** representation learning, embeddings, cosine similarity, nearest-neighbour search, dimensionality reduction.

In [ ]:
feat_model = keras.Model(clf.input, clf.get_layer("gap").output)
feats = feat_model.predict(for_cls(test_ds), verbose=0).astype("float32")
fn = feats / (np.linalg.norm(feats, axis=1, keepdims=True) + 1e-8)
sim = fn @ fn.T; np.fill_diagonal(sim, -1)
nn5 = np.argsort(-sim, axis=1)[:, :5]
p5 = float((y_test[nn5] == y_test[:, None]).mean())
RESULTS["Retrieval (test)"] = {"precision@5": p5}
print(f"Retrieval precision@5 on the test set: {p5:.4f}")

qs = list(range(0, min(len(y_test), 4 * 37), 37))[:4] or [0]
fig, axes = plt.subplots(len(qs), 6, figsize=(13, 2.3 * len(qs))); axes = np.atleast_2d(axes)
for r, q in enumerate(qs):
    row = [q] + nn5[q].tolist()
    for c, i in enumerate(row):
        axes[r, c].imshow(test_np["image"][i]); axes[r, c].axis("off")
        axes[r, c].set_title(("QUERY: " if c == 0 else "") + CLASSES[int(y_test[i])], fontsize=7,
                             color="black" if c == 0 or y_test[i] == y_test[q] else "red")
plt.suptitle("Nearest neighbours in embedding space (red title = different breed)"); plt.tight_layout(); plt.show()

In [ ]:
from sklearn.manifold import TSNE
m_ = min(1500, len(feats))
sel = np.random.default_rng(0).choice(len(feats), m_, replace=False)
emb = TSNE(n_components=2, init="pca", perplexity=min(30, max(5, m_ // 4)), random_state=0).fit_transform(feats[sel])
fig, ax = plt.subplots(1, 2, figsize=(14, 6))
ax[0].scatter(emb[:, 0], emb[:, 1], c=CLASS_IS_CAT[y_test[sel]], cmap="coolwarm", s=6); ax[0].set_title("t-SNE coloured by species (red = cat)")
ax[1].scatter(emb[:, 0], emb[:, 1], c=y_test[sel], cmap="nipy_spectral", s=6); ax[1].set_title("t-SNE coloured by breed")
for a in ax: a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

## 13. Unsupervised / generative: a denoising convolutional autoencoder

No labels at all. The network gets a **noisy** 64x64 image and must reconstruct the **clean** one through a narrow bottleneck (a 4x4x64 latent code), so it learns image structure. We then **interpolate between two latent codes** to see a (blurry) generative model in action. Noise is re-sampled on the fly every epoch.

**Concepts:** autoencoders, bottleneck/latent space, transposed convolutions, denoising, PSNR, latent interpolation (the road to VAEs/GANs/diffusion).

In [ ]:
NOISE = 0.2
def for_ae(ds):
    def f(d):
        clean = tf.image.resize(d["image"] / 255.0, (64, 64), method="area")
        noisy = tf.clip_by_value(clean + tf.random.normal(tf.shape(clean), stddev=NOISE), 0.0, 1.0)
        return noisy, clean
    return ds.map(f, AUTOTUNE)

def psnr_metric(y_true, y_pred):
    return tf.image.psnr(tf.cast(y_true, tf.float32), tf.cast(y_pred, tf.float32), max_val=1.0)

def build_ae(z=64):
    inp = keras.Input((64, 64, 3)); x = inp
    for f in (32, 64, 128):
        x = layers.Conv2D(f, 4, strides=2, padding="same")(x); x = layers.BatchNormalization()(x); x = layers.LeakyReLU(0.2)(x)
    latent = layers.Conv2D(z, 4, strides=2, padding="same", name="latent")(x)                 # 4x4xz bottleneck
    dec_layers = []
    for f in (128, 64, 32):
        dec_layers += [layers.Conv2DTranspose(f, 4, strides=2, padding="same"), layers.BatchNormalization(), layers.ReLU()]
    dec_layers += [layers.Conv2DTranspose(3, 4, strides=2, padding="same", activation="sigmoid", dtype="float32")]
    def run(t):
        for lyr in dec_layers: t = lyr(t)
        return t
    ae_ = keras.Model(inp, run(latent), name="ae")
    enc_ = keras.Model(inp, latent, name="ae_encoder")
    dec_in = keras.Input((4, 4, z)); dec_ = keras.Model(dec_in, run(dec_in), name="ae_decoder")   # shares weights with ae_
    return ae_, enc_, dec_

ae, ae_enc, ae_dec = build_ae()
EP = E(15)
ae.compile(optimizer=cosine_opt(2e-3, EP, wd=0.0), loss="mse", metrics=[psnr_metric])
h_ae = ae.fit(for_ae(train_ds), validation_data=for_ae(val_ds), epochs=EP)
plot_history(h_ae.history, ["loss", "psnr_metric"], "AE")
RESULTS["Denoising AE (val)"] = {"PSNR_dB": float(h_ae.history["val_psnr_metric"][-1])}

In [ ]:
noisy, clean = next(iter(for_ae(test_ds)))
recon = ae.predict(noisy, verbose=0)
n_show = min(8, len(clean))
fig, axes = plt.subplots(3, n_show, figsize=(1.9 * n_show, 5.8))
for j in range(n_show):
    for r, t in enumerate([clean, noisy, recon]):
        axes[r, j].imshow(np.clip(np.asarray(t[j]), 0, 1)); axes[r, j].axis("off")
for r, t in enumerate(["clean", "noisy input", "denoised"]): axes[r, 0].set_title(t, fontsize=9, loc="left")
plt.tight_layout(); plt.show()

# latent interpolation between two images
z = ae_enc.predict(clean[:2], verbose=0); steps = 8
blend = np.concatenate([(1 - a) * z[0:1] + a * z[1:2] for a in np.linspace(0, 1, steps)])
gen = ae_dec.predict(blend, verbose=0)
fig, axes = plt.subplots(1, steps, figsize=(1.9 * steps, 2.2))
for j in range(steps): axes[j].imshow(np.clip(gen[j], 0, 1)); axes[j].axis("off")
plt.suptitle("Latent-space interpolation between two images"); plt.tight_layout(); plt.show()

## 14. Robustness: what happens when images get worse?

Test accuracy on clean images overstates real-world performance. We corrupt the test images at 4 severity levels (**noise, blur, darkening, occlusion, pixelation**) and compare the **ResNet-50** against **ConvNeXt-Tiny**.

**Concepts:** distribution shift, corruption benchmarks (ImageNet-C idea), occlusion, robustness across architectures.

In [ ]:
CORRUPTIONS = {"gaussian_noise": [0.05, 0.1, 0.2, 0.3], "blur": [1.0, 2.0, 3.0, 5.0], "darken": [0.8, 0.6, 0.4, 0.25],
               "occlusion": [0.15, 0.3, 0.45, 0.6], "pixelate": [2, 4, 8, 16]}

def gaussian_blur(x, sigma):
    r = int(3 * sigma); ax = tf.range(-r, r + 1, dtype=tf.float32)
    k1 = tf.exp(-ax ** 2 / (2 * sigma ** 2)); k1 = k1 / tf.reduce_sum(k1)
    k = tf.tile((k1[:, None] * k1[None, :])[:, :, None, None], [1, 1, 3, 1])
    return tf.nn.depthwise_conv2d(x, k, [1, 1, 1, 1], "SAME")

def occlude(x, frac):
    B = tf.shape(x)[0]; k = int(S * frac)
    ys = tf.random.uniform([B], 0, S - k + 1, dtype=tf.int32); xs = tf.random.uniform([B], 0, S - k + 1, dtype=tf.int32)
    rows = tf.range(S)[None, :, None]; cols = tf.range(S)[None, None, :]
    inside = (rows >= ys[:, None, None]) & (rows < ys[:, None, None] + k) & (cols >= xs[:, None, None]) & (cols < xs[:, None, None] + k)
    return tf.where(inside[..., None], 127.5, x)

def corrupt(x, kind, p):                       # x: float32 (B,S,S,3) in 0..255
    if kind == "gaussian_noise": return tf.clip_by_value(x + tf.random.normal(tf.shape(x), stddev=p * 255.0), 0, 255)
    if kind == "blur":           return gaussian_blur(x, p)
    if kind == "darken":         return x * p
    if kind == "occlusion":      return occlude(x, p)
    if kind == "pixelate":
        small = tf.image.resize(x, (S // int(p), S // int(p)), method="area")
        return tf.image.resize(small, (S, S), method="nearest")

n_rob = min(480, len(y_test))
X = test_np["image"][:n_rob].astype("float32"); Y = y_test[:n_rob]

def acc_on(model, kind=None, p=None, bs=48):
    correct = 0
    for i in range(0, len(X), bs):
        xb = tf.constant(X[i:i + bs])
        if kind: xb = corrupt(xb, kind, p)
        correct += int((model.predict_on_batch(xb).argmax(1) == Y[i:i + bs]).sum())
    return correct / len(X)

models_rob = {"ResNet-50": clf, "ConvNeXt-T": convnext}
rob = {m: {k: [acc_on(mod)] + [acc_on(mod, k, p) for p in ps] for k, ps in CORRUPTIONS.items()} for m, mod in models_rob.items()}
fig, axes = plt.subplots(1, len(CORRUPTIONS), figsize=(4 * len(CORRUPTIONS), 3.4))
for ax, k in zip(axes, CORRUPTIONS):
    for m in models_rob: ax.plot(range(5), rob[m][k], marker="o", label=m)
    ax.set_title(k); ax.set_xlabel("severity (0 = clean)"); ax.set_ylim(0, 1.02)
axes[0].set_ylabel("accuracy"); axes[0].legend(); plt.tight_layout(); plt.show()
for m in models_rob:
    RESULTS[f"Robustness {m}"] = {"clean_acc": rob[m]["blur"][0], "mean_acc_corrupted": float(np.mean([v[1:] for v in rob[m].values()]))}
    print(m, RESULTS[f"Robustness {m}"])

## 15. Export, inference on your own photo, and latency

Training is half the job. Here we save the model as **`.keras`** (reloadable Keras format), export a **SavedModel** (for TF Serving), convert to **TFLite** (mobile / edge) and verify numerical parity, **benchmark latency**, and provide a `predict_pet` helper you can point at any image file or URL.

**Concepts:** model serialization, SavedModel vs `.keras` vs TFLite, quantization, preprocessing baked into the graph, latency vs throughput.

In [ ]:
# 1) reload test: the preprocessing layer is baked in, so the loaded model takes raw 0-255 RGB
reloaded = keras.saving.load_model("pet_resnet50_cls.keras", compile=False)
xb = test_np["image"][:4].astype("float32")
print("reload parity (max abs diff):", float(np.abs(reloaded.predict(xb, verbose=0) - clf.predict(xb, verbose=0)).max()))

# 2) SavedModel + TFLite.  Mixed-precision (fp16) layers do not convert to TFLite, so rebuild a clean float32 copy and copy the weights in.
try:
    keras.mixed_precision.set_global_policy("float32")
    fp32, _ = build_transfer_classifier(weights=None)
    fp32.set_weights(clf.get_weights())
    print("float32 copy vs trained model, max abs diff:", float(np.abs(fp32.predict(xb, verbose=0) - clf.predict(xb, verbose=0)).max()))
    with contextlib.redirect_stdout(io.StringIO()):          # Keras prints a very long tensor listing here
        fp32.export("pet_cls_savedmodel")
    conv = tf.lite.TFLiteConverter.from_saved_model("pet_cls_savedmodel")
    conv.optimizations = [tf.lite.Optimize.DEFAULT]          # dynamic-range quantization
    tfl = conv.convert()
    open("pet_resnet50_cls.tflite", "wb").write(tfl)
    try:
        from ai_edge_litert.interpreter import Interpreter          # newer name (pip install ai-edge-litert)
    except Exception:
        Interpreter = tf.lite.Interpreter
    interp = Interpreter(model_content=tfl); interp.allocate_tensors()
    inp_d, out_d = interp.get_input_details()[0], interp.get_output_details()[0]
    interp.set_tensor(inp_d["index"], xb[:1]); interp.invoke()
    tfl_out = interp.get_tensor(out_d["index"])
    print(f"TFLite size: {len(tfl) / 1e6:.1f} MB | argmax agrees with Keras: {tfl_out.argmax() == fp32.predict(xb[:1], verbose=0).argmax()}")
except Exception as e:
    print("SavedModel/TFLite export skipped:", str(e)[:500])
finally:
    if MIXED_PRECISION: keras.mixed_precision.set_global_policy("mixed_float16")

# 3) latency
def bench(model, bs, iters=20):
    x_ = tf.random.uniform((bs, S, S, 3), 0, 255)
    for _ in range(3): model(x_, training=False)
    t0 = time.time()
    for _ in range(iters): _ = model(x_, training=False).numpy()
    return (time.time() - t0) / iters * 1000
print(f"ResNet-50 latency: batch 1 = {bench(clf, 1):.1f} ms | batch 32 = {bench(clf, 32):.1f} ms")

In [ ]:
from urllib.request import urlopen
from PIL import Image

IMAGE_SOURCE = None   # <- put a file path or an http(s) URL of your own pet photo here, e.g. "my_dog.jpg"

def predict_pet(source, topk=5):
    if source.startswith("http"):
        img = Image.open(io.BytesIO(urlopen(source).read())).convert("RGB")
    else:
        img = Image.open(source).convert("RGB")
    arr = np.asarray(img.resize((S, S), Image.BILINEAR), dtype="float32")[None]
    out = multi.predict(arr, verbose=0)
    probs = out["cls"][0]; top = np.argsort(-probs)[:topk]
    fig, ax = plt.subplots(1, 3, figsize=(13, 4))
    ax[0].imshow(arr[0].astype(np.uint8)); draw_box(ax[0], out["box"][0], "red"); ax[0].set_title("input + predicted box"); ax[0].axis("off")
    ax[1].imshow(SEG_COLORS[out["seg"][0].argmax(-1)]); ax[1].set_title("predicted segmentation"); ax[1].axis("off")
    ax[2].barh([CLASSES[i] for i in top][::-1], probs[top][::-1]); ax[2].set_title("top predictions")
    plt.tight_layout(); plt.show()

predict_pet(IMAGE_SOURCE) if IMAGE_SOURCE else print("Set IMAGE_SOURCE above to try your own photo.")

## 16. Final scoreboard

In [ ]:
summary = pd.DataFrame({k: pd.Series(v) for k, v in RESULTS.items()}).T
pd.set_option("display.float_format", lambda v: f"{v:.4f}")
display(summary)
with open("results_tensorflow.json", "w") as f:
    json.dump({k: {a: float(b) for a, b in v.items()} for k, v in RESULTS.items()}, f, indent=2)
print("saved results_tensorflow.json")

### PyTorch <-> Keras cheat-sheet (what you just saw, side by side)

| Idea | PyTorch notebook | This notebook |
|------|------------------|---------------|
| Data loading | `Dataset` + `DataLoader` (Python workers) | `tf.data` graph (`map`, `cache`, `shuffle`, `batch`, `prefetch`) |
| Model definition | subclass `nn.Module`, write `forward` | Functional API (`keras.Input` -> layers -> `keras.Model`) |
| Training loop | hand-written (`train_model`) | `model.compile()` + `model.fit()` |
| Multi-task losses | sum losses manually in `step_fn` | dict of losses / `loss_weights` / metrics keyed by output name |
| Freezing | `p.requires_grad = False` | `layer.trainable = False` + **re-compile** |
| Autograd for Grad-CAM | hooks + `.backward()` | `tf.GradientTape` + `tape.watch` |
| Mixed precision | `torch.autocast` + `GradScaler` | `keras.mixed_precision.set_global_policy` |
| Export | `torch.save`, ONNX | `.keras`, SavedModel, TFLite |

### Where to go next
* **Self-supervised learning** (SimCLR / DINO) on the unlabeled images, then linear-probe the features.
* **Instance segmentation** (Mask R-CNN) and **keypoints / landmarks**; train a real detector with KerasCV / `keras_hub`.
* **Diffusion / GAN** generation, **CLIP** zero-shot classification and text-to-image retrieval.
* **Hyper-parameter search** (KerasTuner / Optuna), **loss-weight ablations** for the multi-task model, **K-fold CV**.
* **Knowledge distillation** and **full-integer quantization** to shrink the model for mobile.